In [1]:
import warnings
warnings.filterwarnings("ignore")
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
import numpy as np 
import pandas as pd


Instructions for updating:
non-resource variables are not supported in the long term


날씨와 농산물 가격은 어떤 상관 관계가 있는지 예측한다.

기상 정보: 기상 자료 개방 포털, https://data.kma.go.kr/cmmn/main.do  
농산물 가격 정보: 농산물 유통 정보, https://www.kamis.or.kr/customer/main/main.do

price_data.csv 파일을 읽어들여 데이터프레임으로 저장한다.  
기상 정보: 평균기온(avgTemp), 최저기온(minTemp), 최고기온(maxTemp), 강수량(rainFall), 가격 정보: 평균가격(avgPrice)

In [2]:
# csv 파일의 첫 줄에 컬럼 이름(헤더)이 없을 경우 header=None 옵션을 지정해서 읽어들인 후 컬럼 이름을 지정하면 된다.
# price_data_df = pd.read_csv('./data/price_data.csv', header=None)
# price_data_df.columns = ['year', 'avgTemp', 'minTemp', 'maxTemp', 'rainFall', 'avgPrice']
price_data_df = pd.read_csv('./data/price_data.csv')
price_data_df

,year,avgTemp,minTemp,maxTemp,rainFall,avgPrice
0,20100101,-4.9,-11.0,0.9,0.0,2123
1,20100102,-3.1,-5.5,5.5,0.8,2123
2,20100103,-2.9,-6.9,1.4,0.0,2123
3,20100104,-1.8,-5.1,2.2,5.9,2020
4,20100105,-5.2,-8.7,-1.8,0.7,2060
...,...,...,...,...,...,...
2917,20171227,-3.9,-8.0,0.7,0.0,2865
2918,20171228,-1.5,-6.9,3.7,0.0,2884
2919,20171229,2.9,-2.1,8.0,0.0,2901
2920,20171230,2.9,-1.6,7.1,0.6,2901


데이터프레임에 저장된 데이터를 numpy 배열로 변환한다.

In [3]:
data = np.array(price_data_df) # price_data_df.values
print(type(data))
print(data)

<class 'numpy.ndarray'>
[[ 2.0100101e+07 -4.9000000e+00 -1.1000000e+01  9.0000000e-01
   0.0000000e+00  2.1230000e+03]
 [ 2.0100102e+07 -3.1000000e+00 -5.5000000e+00  5.5000000e+00
   8.0000000e-01  2.1230000e+03]
 [ 2.0100103e+07 -2.9000000e+00 -6.9000000e+00  1.4000000e+00
   0.0000000e+00  2.1230000e+03]
 ...
 [ 2.0171229e+07  2.9000000e+00 -2.1000000e+00  8.0000000e+00
   0.0000000e+00  2.9010000e+03]
 [ 2.0171230e+07  2.9000000e+00 -1.6000000e+00  7.1000000e+00
   6.0000000e-01  2.9010000e+03]
 [ 2.0171231e+07  2.1000000e+00 -2.0000000e+00  5.8000000e+00
   4.0000000e-01  2.9010000e+03]]


numpy 배열에서 변화 요인(피쳐) 데이터(평균기온, 최저기온, 최고기온, 강수량)로 사용할 데이터를 뽑아낸다. => 학습 데이터

In [4]:
xData = data[:, 1:5]
print(xData)

[[ -4.9 -11.    0.9   0. ]
 [ -3.1  -5.5   5.5   0.8]
 [ -2.9  -6.9   1.4   0. ]
 ...
 [  2.9  -2.1   8.    0. ]
 [  2.9  -1.6   7.1   0.6]
 [  2.1  -2.    5.8   0.4]]


numpy 배열에서 변화 요인에 따른 레이블 데이터(평균가격)로 사용할 데이터를 뽑아낸다. => 실제값

In [5]:
# yData = data[:, -1] # 인덱싱으로 뽑아내면 1차원 형태로 데이터를 뽑아낸다.
# 변화 요인 데이터가 2차원이므로 변화 요인에 따른 레이블 데이터도 2차원이어야 한다. 1차원 형태로 뽑아냈으면 reshape() 메소드로 2차원으로 변환시킨다.
# yData = data[:, -1].reshape(-1, 1)
# yData = data[:, 5:6] # 슬라이싱으로 뽑아내면 2차원 형태로 데이터를 뽑아낸다.
yData = data[:, [-1]] # 인덱싱으로 뽑아낸 결과를 리스트화 시켜도 2차원 형태의 데이터로 뽑아낸다.
print(yData)

[[2123.]
 [2123.]
 [2123.]
 ...
 [2901.]
 [2901.]
 [2901.]]


학습 데이터와 학습 데이터에 따른 레이블을 저장할 placeholder를 선언한다.

In [6]:
X = tf.placeholder(dtype=tf.float64, shape=[None, 4]) # 학습 데이터(변화요인)를 기억할 placeholder를 선언한다.
Y = tf.placeholder(dtype=tf.float64, shape=[None, 1]) # 학습 데이터에 따른 레이블(평균가격)을 기억할 placeholder를 선언한다.

다중 선형 회귀 모델의 가중치와 바이어스를 임의의 값으로 초기화 한다.

In [7]:
a = tf.Variable(tf.random_uniform([4, 1], dtype=tf.float64)) # 가중치, 학습 데이터와 내적하기위해서 4행 1열로 난수를 발생시킨다.
b = tf.Variable(tf.random_uniform([1], dtype=tf.float64)) # 바이어스

sess = tf.Session()
sess.run(tf.global_variables_initializer())

print('가중치\n', sess.run(a), sep='')
print('바이어스\n', sess.run(b), sep='')

가중치
[[0.26724482]
 [0.98852748]
 [0.5720248 ]
 [0.08051116]]
바이어스
[0.08342312]


다중 선형 회귀 모델의 가설(예측값을 계산하는 수식)을 행렬의 내적을 이용해서 만든다.

In [8]:
y = tf.matmul(X, a) + b # 예측값

평균 제곱근 오차(RMSE) 함수를 만든다.

In [9]:
loss = tf.sqrt(tf.reduce_mean(tf.square(y - Y))) # 예측값(y)과 실제값(Y)의 편차에 제곱에 대한 평균을 계산해서 루트를 씌운다.

경사 하강법 알고리즘을 사용해서 오차 함수의 결과를 최소로하는 가중치와 바이어스를 찾는다.

In [10]:
train = tf.train.GradientDescentOptimizer(0.01).minimize(loss)

학습시킨다.

In [11]:
sess = tf.Session()
sess.run(tf.global_variables_initializer())

for epoch in range(100001):
    _, _loss = sess.run([train, loss], feed_dict={X: xData, Y: yData})
    if epoch % 5000 == 0:
        print('epoch: {:6d}, loss: {:10.4f}'.format(epoch, _loss))

epoch:      0, loss:  3520.7909
epoch:   5000, loss:  1733.0313
epoch:  10000, loss:  1565.6364
epoch:  15000, loss:  1521.5146
epoch:  20000, loss:  1511.1455
epoch:  25000, loss:  1508.4255
epoch:  30000, loss:  1507.3550
epoch:  35000, loss:  1506.6376
epoch:  40000, loss:  1505.9971
epoch:  45000, loss:  1505.3748
epoch:  50000, loss:  1504.7583
epoch:  55000, loss:  1504.1450
epoch:  60000, loss:  1503.5344
epoch:  65000, loss:  1502.9262
epoch:  70000, loss:  1502.3204
epoch:  75000, loss:  1501.7171
epoch:  80000, loss:  1501.1162
epoch:  85000, loss:  1500.5177
epoch:  90000, loss:  1499.9217
epoch:  95000, loss:  1499.3280
epoch: 100000, loss:  1498.7367


테스트

In [12]:
newData = [[3.2, -2.5, 7.1, 15.0]]
result = sess.run(y, feed_dict={X: newData})
print('{}'.format(result))
print('{:,.0f}'.format(result[0][0]))

[[2970.09580189]]
2,970


학습 완료된 예측 모델을 디스크에 저장한다.

In [13]:
# Saver() 클래스 객체는 학습된 모델을 디스크에 저장하거나 불러올때 사용한다.
saver = tf.train.Saver()
# Saver() 클래스 객체의 save() 메소드로 학습된 모델을 디스크에 저장한다.
saver.save(sess, './model/baechu.ckpt')
print('학습된 모델을 저장했습니다.')

학습된 모델을 저장했습니다.


In [14]:
sess.close()